# Query Agent Evaluation with Strands Framework

This notebook evaluates the Virtual Knowledge Graph Query Agent using the AWS Strands evaluation framework.

## AWS Credentials Setup

The ontology query agent needs AWS credentials to access:
- **Amazon Bedrock** - For Claude model inference (`global.anthropic.claude-sonnet-4-6`)
- **Amazon Athena** - For executing SQL queries (`execute_sql_query`)
- **Amazon SSM Parameter Store** - For retrieving the Athena results bucket name
- **Amazon Bedrock AgentCore Gateway** - For Neptune tool access (`get_ontology_from_neptune`, `execute_sparql_query`)
- **Amazon DynamoDB** - For reading ontology metadata by `id`

### Credential Injection

This notebook uses an AWS profile configured locally. The boto3 session is created with this profile and then injected into the ontology query agent using `set_boto_session()`. This ensures all AWS service calls within the agent use the same credentials.

### Verification Steps

1. Import agent module and utilities
2. Create boto3 session with AWS profile
3. Verify credentials with STS `get_caller_identity()`
4. Inject session into agent using `set_boto_session()`
5. Configure `EVAL_ONTOLOGY_ID` (DynamoDB key for the target ontology)
6. Run evaluations

In [1]:
# !pip install strands-agents strands-agents-tools strands-agents-evals python-dotenv --quiet

In [2]:
# Import required libraries
import sys
import os
import json
import pandas as pd
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, BotoCoreError
from datetime import datetime
from dotenv import load_dotenv

# Add ontology query agent to path
sys.path.append('../agents')

# Import agent functions
from ontology_query_agent.main import invoke, reset_agent_state, set_boto_session

# Configure boto3 session with AWS profile
config = Config(
    retries={
        'max_attempts': 10,
        'mode': 'standard'
    },
    signature_version='v4'
)
# Load environment variables from .env file (falls back to os.environ if not found)
load_dotenv(dotenv_path='.env', override=False)

# Create session with specific AWS profile
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'XXX'))
region = session.region_name or 'us-east-1'

# Create STS client to verify credentials
sts = session.client(
    service_name='sts',
    region_name=region,
    config=config
)

# Verify credentials work
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ.get('AWS_PROFILE', 'XXX')}")
    print(f"  Account: ...{identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

# Inject session into query agent
set_boto_session(session)
print(f"\n✓ Agent configured to use boto3 session credentials")
print(f"  All AWS service calls (Bedrock, Athena, Neptune Gateway, SSM) will use this session")

# Load job_id stored by ontology_agent_evaluation.ipynb via %store
%store -r job_id
ONTOLOGY_ID = job_id
print(f"\n✓ Loaded ontology_id from %%store: {ONTOLOGY_ID}")
print(f"All evals will use Ontology_ID=....{ONTOLOGY_ID[-4:]}")

2026-03-16 09:41:21,052 - ontology_query_agent.main - INFO - Boto3 session set with region: us-east-1


✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1

✓ Agent configured to use boto3 session credentials
  All AWS service calls (Bedrock, Athena, Neptune Gateway, SSM) will use this session

✓ Loaded ontology_id from %%store: single_table_basic-d7b2932e
All evals will use Ontology_ID=....932e


In [ ]:
# S3 Tables (Iceberg) data source config — loaded from .env
# Same pattern as metadata_agent_evaluation.ipynb
S3T_CATALOG  = os.environ.get('S3T_CATALOG', 'XXX')
S3T_DATABASE = os.environ.get('S3T_DATABASE', 'XXX')

print("✓ Catalog/database configured:")
print(f"  S3T_CATALOG  = {S3T_CATALOG}")
print(f"  S3T_DATABASE = {S3T_DATABASE}")

## Load Evaluation Dataset

In [3]:
# Load evaluation dataset
with open('../data/eval/ontology_query_agent_evaluation_dataset.json', 'r') as f:
    test_cases = json.load(f)

print(f"Loaded {len(test_cases)} test cases")
print(f"\nCategories: {set(tc['category'] for tc in test_cases)}")

Loaded 1 test cases

Categories: {'admincodes_test1'}


## Setup Strands Evaluation Framework

Configure telemetry and evaluation infrastructure

In [ ]:
# Convert test cases to Strands Case objects
strands_cases = []

for test_case in test_cases:
    case = Case[str, str](
        name=test_case['id'],
        input=test_case['query'],
        expected_output=test_case['expected'],
        metadata={
            'category': test_case['category'],
            'id': ONTOLOGY_ID,
            'expected_tools': test_case.get('expected_tools', []),
            'expected_tables': test_case.get('expected_tables', []),
            'database': S3T_DATABASE,
            'catalog_id': S3T_CATALOG,
        }
    )
    strands_cases.append(case)

print(f"✓ Converted {len(strands_cases)} test cases to Strands Case format")
print(f"\nCategories:")
categories = {}
for case in strands_cases:
    cat = case.metadata['category']
    categories[cat] = categories.get(cat, 0) + 1

for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} cases")

In [5]:
# Convert test cases to Strands Case objects
strands_cases = []

for test_case in test_cases:
    case = Case[str, str](
        name=test_case['id'],
        input=test_case['query'],
        expected_output=test_case['expected'],
        metadata={
            'category': test_case['category'],
            'id': ONTOLOGY_ID,
            'expected_tools': test_case.get('expected_tools', []),
            'expected_tables': test_case.get('expected_tables', []),
            'database': test_case.get('database', 'default'),
            'catalog_id': test_case.get('catalog_id', '')
        }
    )
    strands_cases.append(case)

print(f"✓ Converted {len(strands_cases)} test cases to Strands Case format")
print(f"\nCategories:")
categories = {}
for case in strands_cases:
    cat = case.metadata['category']
    categories[cat] = categories.get(cat, 0) + 1

for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} cases")

✓ Converted 1 test cases to Strands Case format

Categories:
  admincodes_test1: 1 cases


## Define Task Function

Create task function that captures agent execution traces for evaluation

In [6]:
def query_agent_task(case: Case) -> dict:
    """
    Task function that executes the ontology query agent and captures telemetry.

    This function:
    1. Clears previous telemetry data
    2. Resets agent state
    3. Invokes the agent with the test query via invoke(payload)
    4. Captures execution spans
    5. Maps spans to a session for trajectory evaluation

    The invoke() payload requires:
      - 'question': the natural language query
      - 'id': the DynamoDB ontology ID (used to look up the ontology name, which is
               prepended to the question as "[ontology: <name>]")
      - 'catalog_id': the Athena catalog ID (e.g. 's3tablescatalog/semantic-layer-analytics-tables')

    Returns:
        dict: Contains 'output' (agent answer) and 'trajectory' (session spans)
    """
    # Clear previous spans
    memory_exporter.clear()

    # Reset agent state for clean test
    reset_agent_state()

    try:
        # Invoke the query agent
        start_time = datetime.now()
        payload = {
            "question": case.input,
            "id": case.metadata.get('id', ONTOLOGY_ID),
            "catalog_id": case.metadata.get('catalog_id', ''),
        }
        response = invoke(payload)
        duration = (datetime.now() - start_time).total_seconds()

        # Extract answer text from structured response
        # invoke() returns: answer, sql_query, results, n_quads, reasoning, metadata
        if isinstance(response, dict):
            if response.get('needs_clarification'):
                response_text = response.get('clarification_question', str(response))
            else:
                response_text = response.get('answer', str(response))
        else:
            response_text = str(response)

        # Get captured spans from telemetry
        finished_spans = list(memory_exporter.get_finished_spans())

        # Map spans to session for trajectory-based evaluators
        mapper = StrandsInMemorySessionMapper()
        session_obj = mapper.map_to_session(finished_spans, session_id=case.session_id)

        return {
            "output": response_text,
            "trajectory": session_obj,
            "duration": duration,
            "success": True,
            "error": None
        }

    except Exception as e:
        print(f"✗ Error in {case.name}: {str(e)}")
        return {
            "output": "",
            "trajectory": None,
            "duration": 0,
            "success": False,
            "error": str(e)
        }

print("✓ Task function defined")
print("  Function: query_agent_task()")
print(f"  Payload: 'question': ..., 'id': {ONTOLOGY_ID}, 'catalog_id': ...")
print("  Captures: Agent answer + execution traces")

✓ Task function defined
  Function: query_agent_task()
  Payload: 'question': ..., 'id': single_table_basic-d7b2932e, 'catalog_id': ...
  Captures: Agent answer + execution traces


## Create Built-in Evaluators

Configure Strands evaluators for multi-dimensional assessment

In [7]:
from strands.models import BedrockModel

# Create judge model using same boto session
judge_model = BedrockModel(
    model_id='global.anthropic.claude-sonnet-4-6',
    temperature=0.0,
    boto_session=session
)

# Initialize built-in evaluators
evaluators = [
    # TRACE_LEVEL: Response quality evaluation
    HelpfulnessEvaluator(model=judge_model),

    # TRACE_LEVEL: Faithfulness to context (detects hallucinations)
    FaithfulnessEvaluator(model=judge_model),

    # TOOL_LEVEL: Validate tool selection (Neptune Gateway, Athena)
    ToolSelectionAccuracyEvaluator(model=judge_model),

    # TOOL_LEVEL: Validate tool parameters (SQL queries, ontology name, etc.)
    ToolParameterAccuracyEvaluator(model=judge_model),

    # SESSION_LEVEL: Overall goal achievement
    GoalSuccessRateEvaluator(model=judge_model)
]

print(f"✓ Created {len(evaluators)} evaluators:")
print(f"  1. HelpfulnessEvaluator           - User satisfaction & utility")
print(f"  2. FaithfulnessEvaluator          - Groundedness & hallucination detection")
print(f"  3. ToolSelectionAccuracyEvaluator - Correct tool choice")
print(f"  4. ToolParameterAccuracyEvaluator - Tool parameter validation")
print(f"  5. GoalSuccessRateEvaluator       - Overall success rate")
print(f"\n  All evaluators use claude-sonnet-4-6 as judge")

✓ Created 5 evaluators:
  1. HelpfulnessEvaluator           - User satisfaction & utility
  2. FaithfulnessEvaluator          - Groundedness & hallucination detection
  3. ToolSelectionAccuracyEvaluator - Correct tool choice
  4. ToolParameterAccuracyEvaluator - Tool parameter validation
  5. GoalSuccessRateEvaluator       - Overall success rate

  All evaluators use claude-sonnet-4-6 as judge


## Run Evaluation Experiment

Execute test cases and evaluate with all built-in evaluators

In [ ]:
# Create experiment with test cases and evaluators
experiment = Experiment[str, str](
    cases=strands_cases,
    evaluators=evaluators
)

print(f"Starting experiment with {len(strands_cases)} test cases...")
print(f"{'='*70}\n")

# Run evaluations
reports = experiment.run_evaluations(query_agent_task)

print(f"\n{'='*70}")
print(f"✓ Evaluation complete!")
print(f"  Generated {len(reports)} evaluation reports")

2026-03-16 09:41:21,190 - ontology_query_agent.main - INFO - Agent state reset for session: None
2026-03-16 09:41:21,191 - ontology_query_agent.main - INFO - Agent state reset for session: d6a15c24


Starting experiment with 1 test cases...



2026-03-16 09:41:21,413 - ontology_query_agent.main - INFO - Neptune Gateway configured - using Gateway for Neptune tools (IAM auth)
2026-03-16 09:41:21,414 - ontology_query_agent.main - INFO - Neptune Gateway MCP client configured: https://semantic-layer-neptune-gateway-pt7i4au20m.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
/Users/huthmac/.pyenv/versions/3.12.8/lib/python3.12/contextlib.py:105: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)
2026-03-16 09:41:21,769 - httpx - INFO - HTTP Request: POST https://semantic-layer-neptune-gateway-pt7i4au20m.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 200 OK"
2026-03-16 09:41:21,771 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-06-18
2026-03-16 09:41:21,882 - httpx - INFO - HTTP Request: POST https://semantic-layer-neptune-gateway-pt7i4au20m.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 202 Accepted"
2026-03-16 09:41:22,108 - ht

## Aggregate Metrics

Analyze scores across all test cases and evaluators

In [ ]:
# Aggregate evaluation scores across all test cases
# reports is a list[EvaluationReport], one per evaluator (same order as `evaluators` list)
# Each EvaluationReport has: .cases[i] (dict), .scores[i], .test_passes[i], .reasons[i]

evaluation_data = []

if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        query = case_dict.get('input', '')
        case_data = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
            'query': query[:50] + '...' if len(query) > 50 else query,
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_data[eval_name] = report.scores[j] if j < len(report.scores) else None
        evaluation_data.append(case_data)

df_evals = pd.DataFrame(evaluation_data)

print("=== EVALUATION SCORES BY TEST CASE ===\n")
print(df_evals.to_string(index=False))

print("\n=== AVERAGE SCORES BY EVALUATOR ===\n")
evaluator_cols = [col for col in df_evals.columns if col not in ['test_id', 'category', 'query']]
avg_scores = df_evals[evaluator_cols].mean()
print(avg_scores.to_string())

print("\n=== SCORES BY CATEGORY ===\n")
category_scores = df_evals.groupby('category')[evaluator_cols].mean()
print(category_scores.to_string())

## Pass/Fail Analysis

Identify which test cases passed or failed each evaluator

In [ ]:
# Analyze pass/fail status for each evaluator
pass_fail_data = []

if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        case_pass_fail = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_pass_fail[eval_name] = report.test_passes[j] if j < len(report.test_passes) else None
        pass_fail_data.append(case_pass_fail)

df_pass_fail = pd.DataFrame(pass_fail_data)

print("=== PASS/FAIL BY TEST CASE ===\n")
print(df_pass_fail.to_string(index=False))

print("\n=== PASS RATE BY EVALUATOR ===\n")
evaluator_cols = [col for col in df_pass_fail.columns if col not in ['test_id', 'category']]
pass_rates = df_pass_fail[evaluator_cols].mean()
print(pass_rates.to_string())

print("\n=== PASS RATE BY CATEGORY ===\n")
category_pass_rates = df_pass_fail.groupby('category')[evaluator_cols].mean()
print(category_pass_rates.to_string())

## Save Evaluation Results

Export detailed evaluation data for further analysis

In [ ]:
# Save detailed evaluation results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save scores DataFrame
scores_file = f'../data/eval/results/query_agent_evaluation_scores_{timestamp}.csv'
df_evals.to_csv(scores_file, index=False)
print(f"✓ Scores saved to: {scores_file}")

# Save pass/fail DataFrame
pass_fail_file = f'../data/eval/results/query_agent_evaluation_pass_fail_{timestamp}.csv'
df_pass_fail.to_csv(pass_fail_file, index=False)
print(f"✓ Pass/fail data saved to: {pass_fail_file}")

# Save detailed JSON with per-evaluator reasoning
# reports[i] = EvaluationReport for evaluators[i]
# Each report has: .cases[j] (dict), .scores[j], .test_passes[j], .reasons[j]
detailed_results = []

if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        case_result = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
            'query': case_dict.get('input', ''),
            'expected': case_dict.get('expected_output', ''),
            'evaluations': {},
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_result['evaluations'][eval_name] = {
                'score': report.scores[j] if j < len(report.scores) else None,
                'test_pass': report.test_passes[j] if j < len(report.test_passes) else None,
                'reasoning': report.reasons[j] if j < len(report.reasons) else None,
            }
        detailed_results.append(case_result)

detailed_file = f'../data/eval/results/query_agent_evaluation_detailed_{timestamp}.json'
with open(detailed_file, 'w') as f:
    json.dump(detailed_results, f, indent=2)
print(f"✓ Detailed results with reasoning saved to: {detailed_file}")

print(f"\n{'='*70}")
print(f"✓ All evaluation results saved")
print(f"  Scores:    {scores_file}")
print(f"  Pass/Fail: {pass_fail_file}")
print(f"  Detailed:  {detailed_file}")

## Display Detailed Report

View comprehensive evaluation metrics for a sample test case

In [ ]:
# Display detailed report for each evaluator
if reports:
    for k, report in enumerate(reports):
        eval_name = evaluators[k].get_type_name()
        print(f"\n{'='*70}")
        print(f"=== REPORT: {eval_name} (overall_score={report.overall_score:.2f}) ===\n")
        report.display(
            include_input=True,
            include_actual_output=True,
            include_expected_output=True,
        )
else:
    print("No reports generated")

## Summary

### Agent Architecture
The ontology query agent converts natural language questions to SQL using ontology mappings stored in Neptune, executes queries via Athena, and returns semantic RDF results. Key workflow:

1. **`get_ontology_from_neptune(ontology_name)`** — retrieves classes, properties, and `mapsToTable`/`mapsToColumn` mappings from Neptune via AgentCore Gateway
2. **`disambiguate_query_terms(user_query, ontology_info)`** — maps natural language terms to ontology classes and Athena tables
3. **`execute_sql_query(sql_query, database_name, catalog_id)`** — runs SQL on Athena and stores full result set in S3
4. **`map_sql_results_to_rdf(query_results, ontology_info)`** — converts rows to N-Quads using ontology property mappings

### Multi-Dimensional Evaluation
- **HelpfulnessEvaluator**: Measures user satisfaction (7-level scale)
- **FaithfulnessEvaluator**: Detects hallucinations and groundedness issues (5-level scale)
- **ToolSelectionAccuracyEvaluator**: Validates correct tool choice and sequencing
- **ToolParameterAccuracyEvaluator**: Checks SQL query parameters and ontology name handling
- **GoalSuccessRateEvaluator**: Measures end-to-end goal achievement

### Required Environment Variables
- `EVAL_ONTOLOGY_ID` — DynamoDB ontology ID to test against (default: `insurance-demo`)
- `NEPTUNE_GATEWAY_URL` — AgentCore Gateway endpoint for Neptune tools
- `AWS_REGION` — AWS region (default: `us-east-1`)
- `METADATA_TABLE` — DynamoDB table name (default: `semantic-layer-metadata`)

### Next Steps
- Review failed test cases to understand agent weaknesses
- Compare scores across categories (finance, reporting, edge_case, etc.)
- Use tool-level evaluators to debug SQL generation or disambiguation issues
- Track metrics over time as agent improves